# Demand Planner: весь магазин CA_1

В первом ноутбуке проверили прогноз на 100 товарах FOODS. Здесь берём все товары CA_1: продукты, товары для дома и хобби.

Пока готовим данные и считаем простые прогнозы. Для CatBoost сохраним те же признаки, что в первом эксперименте. Новые признаки будем проверять после первого обучения на расширенном наборе.

Разделы 0–4 выполнены. Следующий раздел — обучение CatBoost на расширенной выборке.

## 0. Окружение

Запускаем в `.venv` проекта. Основные расчёты теперь находятся в `src`, а в ноутбуке остаются настройки и результаты.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import sys

import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [current_dir, *current_dir.parents]
     if (path / "requirements.txt").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Откройте ноутбук из папки demand-planner или notebooks.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Выберите Python из окружения .venv проекта.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import download_m5, load_subset, build_weekly_table
from src.features import (
    BASELINE_METHODS, FEATURE_COLUMNS, NUMERIC_FEATURES,
    add_weekly_features, split_by_time,
)
from src.backtest import evaluate_forecasts

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
STORE_ID = "CA_1"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)

print(f"Python: {sys.version.split()[0]}")
print(f"Проект: {PROJECT_ROOT}")
print(f"Магазин: {STORE_ID}")

Python: 3.12.14
Проект: /Users/daniilsestov/Desktop/pet_projects/demand-planner
Магазин: CA_1


## 1. Все товары магазина

Используем те же исходные CSV. `category=None` оставляет все категории, `n_items=None` — все подходящие товары. Случайной выборки в этом запуске нет.

In [2]:
display(download_m5(DATA_RAW))

started = perf_counter()
calendar, sales, prices, source_shapes = load_subset(
    DATA_RAW,
    store_id=STORE_ID,
    category=None,
    n_items=None,
    seed=RANDOM_STATE,
)
load_seconds = perf_counter() - started

assert sales["store_id"].eq(STORE_ID).all()
category_items = sales.groupby("cat_id").agg(items=("item_id", "nunique"))
day_columns = [column for column in sales.columns if column.startswith("d_")]

display(category_items)
print(f"Всего товаров: {len(sales):,}")
print(f"Дней истории: {len(day_columns):,}")
print(f"Чтение данных: {load_seconds:.2f} сек.")

,file,bytes,md5
0,calendar.csv,103469,3ffeab2991b0c8e861d008b39ea4c95c
1,sales_train_validation.csv,120007726,26a366a25beb57b0a8f4c7b148758f94
2,sell_prices.csv,203395785,08c591caa99e55daf3e0ccac913f7c85


,items
cat_id,
FOODS,1437
HOBBIES,565
HOUSEHOLD,1047


Всего товаров: 3,049
Дней истории: 1,913
Чтение данных: 3.31 сек.


В продажах по-прежнему одна строка на товар, дни находятся в отдельных столбцах. Цены загружены только для CA_1. Проверим размеры таблиц и посмотрим несколько идентификаторов.

In [3]:
loaded_tables = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": frame.shape[1],
     "memory_mb": frame.memory_usage(deep=True).sum() / 1024**2}
    for name, frame in [("calendar", calendar), ("sales", sales), ("prices", prices)]
])

display(loaded_tables.round(2))
display(sales[["item_id", "dept_id", "cat_id", "store_id"]].head())

,table,rows,columns,memory_mb
0,calendar,1969,14,0.24
1,sales,3049,1919,22.58
2,prices,698412,4,29.82


,item_id,dept_id,cat_id,store_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1


## 2. Недельные продажи

Полная дневная таблица этого магазина содержала бы почти 5,8 млн строк. Сейчас она нам не нужна: складываем дни по полным неделям прямо из широкой таблицы.

Неделя всё так же идёт с понедельника по воскресенье. Неполные недели исключаем. Нули сохраняем. `first_price_date` — первая дата с записью о цене в доступной истории; это не данные об остатках.

In [4]:
started = perf_counter()
weekly = build_weekly_table(calendar, sales, prices)
weekly_seconds = perf_counter() - started

assert weekly["item_id"].nunique() == sales["item_id"].nunique()
assert len(weekly) == len(sales) * weekly["week_end"].nunique()

display(weekly[[
    "week_end", "item_id", "cat_id", "weekly_sales", "first_price_date"
]].head(10))
print(f"Полных недель: {weekly['week_end'].nunique()}")
print(f"Строк товар-неделя: {len(weekly):,}")
print(f"Память недельной таблицы: {weekly.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")
print(f"Подготовка недель: {weekly_seconds:.2f} сек.")

,week_end,item_id,cat_id,weekly_sales,first_price_date
0,2011-02-06,FOODS_1_001,FOODS,9,2011-01-29
1,2011-02-13,FOODS_1_001,FOODS,7,2011-01-29
2,2011-02-20,FOODS_1_001,FOODS,10,2011-01-29
3,2011-02-27,FOODS_1_001,FOODS,14,2011-01-29
4,2011-03-06,FOODS_1_001,FOODS,16,2011-01-29
5,2011-03-13,FOODS_1_001,FOODS,12,2011-01-29
6,2011-03-20,FOODS_1_001,FOODS,3,2011-01-29
7,2011-03-27,FOODS_1_001,FOODS,8,2011-01-29
8,2011-04-03,FOODS_1_001,FOODS,1,2011-01-29
9,2011-04-10,FOODS_1_001,FOODS,8,2011-01-29


Полных недель: 273
Строк товар-неделя: 832,377
Память недельной таблицы: 23.19 МБ
Подготовка недель: 0.31 сек.


Сохраняем недельную таблицу отдельно от данных первого эксперимента. Строковые идентификаторы хранятся как категории, чтобы повторяющиеся названия занимали меньше памяти.

In [5]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
weekly_path = DATA_PROCESSED / f"m5_{STORE_ID.lower()}_all_weekly.parquet"
weekly.to_parquet(weekly_path, index=False)

display(weekly.groupby("cat_id", observed=True).agg(
    items=("item_id", "nunique"),
    weekly_rows=("weekly_sales", "size"),
    mean_weekly_sales=("weekly_sales", "mean"),
).round(2))
print(f"Сохранено: {weekly_path.name}")

,items,weekly_rows,mean_weekly_sales
cat_id,,,
FOODS,1437,392301,13.70
HOBBIES,565,154245,5.68
HOUSEHOLD,1047,285831,5.04


Сохранено: m5_ca_1_all_weekly.parquet


## 3. Признаки и периоды проверки

Считаем прежние лаги 1–4 недель, средние за 4 и 8 недель, стандартное отклонение за 4 недели, месяц и номер недели. Перед скользящими расчётами стоит сдвиг на одну неделю.

Для сравнения сохраняем даты первого эксперимента. Последние четыре недели называются `reference`: мы уже смотрели их результаты на 100 товарах. Для дальнейшего подбора моделей потребуется отдельно подготовить финальную проверку.

In [6]:
started = perf_counter()
model_data = split_by_time(add_weekly_features(weekly))
feature_seconds = perf_counter() - started

split_summary = model_data.groupby("split", observed=True).agg(
    first_week_end=("week_end", "min"),
    last_week_end=("week_end", "max"),
    weeks=("week_end", "nunique"),
    rows=("weekly_sales", "size"),
)

display(split_summary)
print(f"Подготовка признаков: {feature_seconds:.2f} сек.")
print(f"Память таблицы с признаками: {model_data.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")

,first_week_end,last_week_end,weeks,rows
split,,,,
train,2011-02-06,2016-01-31,261,795789
validation,2016-02-07,2016-03-27,8,24392
reference,2016-04-03,2016-04-24,4,12196


Подготовка признаков: 3.39 сек.
Память таблицы с признаками: 96.22 МБ


Как и в первом ноутбуке, для модели берём периоды, когда к дате прогноза у товара уже была запись о цене. В начале обучения также нужны восемь прошлых недель для `mean_8w`.

Ниже показываем число подходящих строк. Если цена появилась только в течение прогнозируемой недели, такая неделя пока не входит в проверку. Прогноз запуска новых товаров нужно будет рассмотреть отдельно.

In [7]:
has_history = model_data[NUMERIC_FEATURES].notna().all(axis=1)
train_mask = model_data["split"].eq("train") & model_data["known_at_origin"] & has_history
validation_candidates = model_data["split"].eq("validation")
validation_mask = validation_candidates & model_data["known_at_origin"]
validation_data = model_data.loc[validation_mask].copy()

assert validation_data[FEATURE_COLUMNS].notna().all().all()
assert validation_data[list(BASELINE_METHODS.values())].notna().all().all()

print(f"Строк для будущего обучения: {train_mask.sum():,}")
print(f"Всего строк валидации: {validation_candidates.sum():,}")
print(f"Из них с известной ранее ценой: {len(validation_data):,}")
print(f"Исключено до первой известной цены: {(validation_candidates & ~model_data['known_at_origin']).sum():,}")

display(validation_data[[
    "week_end", "item_id", "cat_id", "lag_1w", "mean_4w", "mean_8w", "weekly_sales"
]].head(10).round(2))

Строк для будущего обучения: 623,292
Всего строк валидации: 24,392
Из них с известной ранее ценой: 24,390
Исключено до первой известной цены: 2


/var/folders/_5/xfqsygpn717bwxf2_1nbmcvh0000gn/T/ipykernel_29607/384630940.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].head(10).round(2))


,week_end,item_id,cat_id,lag_1w,mean_4w,mean_8w,weekly_sales
261,2016-02-07,FOODS_1_001,FOODS,7.0,3.75,4.62,8
262,2016-02-14,FOODS_1_001,FOODS,8.0,4.75,4.00,6
263,2016-02-21,FOODS_1_001,FOODS,6.0,5.50,4.38,5
264,2016-02-28,FOODS_1_001,FOODS,5.0,6.50,4.50,1
265,2016-03-06,FOODS_1_001,FOODS,1.0,5.00,4.38,4
266,2016-03-13,FOODS_1_001,FOODS,4.0,4.00,4.38,4
267,2016-03-20,FOODS_1_001,FOODS,4.0,3.50,4.50,10
268,2016-03-27,FOODS_1_001,FOODS,10.0,4.75,5.62,6
534,2016-02-07,FOODS_1_002,FOODS,1.0,1.50,2.75,8
535,2016-02-14,FOODS_1_002,FOODS,8.0,3.00,3.12,3


## 4. Простые прогнозы на полном магазине

Снова сравним прошлую неделю, среднее и медиану четырёх недель. Все методы оцениваем на одинаковых строках валидации.

Здесь изменился состав товаров: добавились другие категории. Поэтому новый MAE нельзя напрямую считать улучшением или ухудшением относительно 100 товаров FOODS. Сравнивать нужно методы на одной выборке.

In [8]:
baseline_results = evaluate_forecasts(validation_data, BASELINE_METHODS)
display(baseline_results.style.format({
    "MAE": "{:.2f}", "WAPE": "{:.2%}", "Bias": "{:+.2%}",
}))

,MAE,WAPE,Bias
Метод,,,
Предыдущая неделя,4.00,39.33%,-1.16%
Среднее за 4 недели,3.56,34.98%,-2.07%
Медиана за 4 недели,3.62,35.57%,-4.27%


Посмотрим ошибки отдельно для FOODS, HOBBIES и HOUSEHOLD. У категорий разный уровень продаж, поэтому общий результат может скрывать различия между ними.

In [9]:
category_results = pd.concat(
    {
        category: evaluate_forecasts(group, BASELINE_METHODS)
        for category, group in validation_data.groupby("cat_id", observed=True)
    },
    names=["category", "Метод"],
)

display(category_results.style.format({
    "MAE": "{:.2f}", "WAPE": "{:.2%}", "Bias": "{:+.2%}",
}))

Сохраняем таблицы и сведения о размере запуска. Здесь пока только данные и baselines; CatBoost на полном магазине ещё не обучен.

In [10]:
scaling_results_dir = RESULTS / "ca1_scaling"
scaling_results_dir.mkdir(parents=True, exist_ok=True)
baseline_results.to_csv(scaling_results_dir / "baseline_metrics.csv", index_label="Метод")
category_results.to_csv(scaling_results_dir / "baseline_metrics_by_category.csv")
category_items.to_csv(scaling_results_dir / "category_items.csv")

scaling_summary = {
    "store_id": STORE_ID,
    "items": len(sales),
    "daily_history_days": len(day_columns),
    "complete_weeks": int(weekly["week_end"].nunique()),
    "weekly_rows": len(weekly),
    "training_rows_eligible": int(train_mask.sum()),
    "validation_rows_available": int(validation_candidates.sum()),
    "validation_rows_evaluated": len(validation_data),
    "selection_rule": "price registered by forecast origin; historical zeros retained",
    "reference_period_already_viewed_in_pilot": True,
    "weekly_memory_mb": float(weekly.memory_usage(deep=True).sum() / 1024**2),
    "feature_memory_mb": float(model_data.memory_usage(deep=True).sum() / 1024**2),
    "load_seconds": load_seconds,
    "weekly_seconds": weekly_seconds,
    "feature_seconds": feature_seconds,
    "catboost_trained": False,
}
(scaling_results_dir / "data_summary.json").write_text(
    json.dumps(scaling_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Результаты: {scaling_results_dir}")

Результаты: /Users/daniilsestov/Desktop/pet_projects/demand-planner/results/ca1_scaling


В CA_1 получилось 3 049 товаров и 832 377 недельных строк. Из 24 392 строк выбранных восьми недель оцениваем 24 390: у двух строк FOODS_3_595 цена ещё не была зарегистрирована к началу прогноза. Обе строки имеют нулевые продажи; правило отбора задано по доступности цены.

Лучший простой метод — среднее четырёх недель: MAE 3,56 и WAPE 34,98%. По категориям MAE составляет 4,76 для FOODS, 2,83 для HOBBIES и 2,30 для HOUSEHOLD. При этом у HOBBIES самая большая относительная ошибка: WAPE 44,11%.

CatBoost предстоит сравнить с этими результатами на тех же строках. Пока перенесены прежние признаки; новые признаки и другие модели здесь ещё не проверялись.

## 5. CatBoost

Обучаем одну модель на всех 3 049 товарах CA_1. Признаки те же: идентификаторы товара и магазина, прошлые продажи и календарь. Ниже готовим данные, обучаем CatBoost и сравниваем его с простыми прогнозами на тех же восьми неделях.


In [22]:
import numpy as np
from catboost import CatBoostRegressor
from src.features import CATEGORICAL_FEATURES

X_train = model_data.loc[train_mask, FEATURE_COLUMNS].copy()
y_train = model_data.loc[train_mask, "weekly_sales"]

X_valid = validation_data[FEATURE_COLUMNS].copy()
y_valid = validation_data["weekly_sales"]

assert X_train.notna().all().all()
assert X_valid.notna().all().all()
assert (
    model_data.loc[train_mask, "week_end"].max()
    < validation_data["week_end"].min()
)

for frame in (X_train, X_valid):
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].astype(str).astype(object)

print(f"Строк обучения: {len(X_train):,}")
print(f"Строк валидации: {len(X_valid):,}")


Строк обучения: 623,292
Строк валидации: 24,390


В первом запуске поставили лимит 1 000 деревьев. Лучший результат оказался на последней итерации, поэтому увеличили лимит до 5 000. Остальные настройки оставили прежними.

Обучение остановится после 100 итераций без улучшения MAE на валидации. В итоговой модели останутся деревья до лучшей итерации. Здесь `test` в логе означает валидацию; период `reference` в обучение и подбор числа деревьев не входит.


In [23]:
model = CatBoostRegressor(
    iterations=5000,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=RANDOM_STATE,
    thread_count=4,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model.tree_count_}")


0:	learn: 9.7751480	test: 8.4068994	best: 8.4068994 (0)	total: 89.3ms	remaining: 7m 26s
250:	learn: 4.0587909	test: 3.4201749	best: 3.4201295 (247)	total: 15.7s	remaining: 4m 56s
500:	learn: 4.0432074	test: 3.4134053	best: 3.4134051 (499)	total: 29.2s	remaining: 4m 22s
750:	learn: 4.0293361	test: 3.3994857	best: 3.3994854 (748)	total: 42s	remaining: 3m 57s
1000:	learn: 3.9784435	test: 3.3780495	best: 3.3780495 (1000)	total: 54.8s	remaining: 3m 39s
1250:	learn: 3.9464084	test: 3.3735920	best: 3.3727405 (1187)	total: 1m 8s	remaining: 3m 23s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.372740497
bestIteration = 1187

Shrink model to first 1188 iterations.
Деревьев в модели: 1188


Считаем прогнозы и общие метрики. Отрицательные прогнозы заменяем нулём: продажи не могут быть отрицательными. Сравниваем MAE, WAPE и суммарное смещение прогноза — Bias.


In [24]:
validation_predictions = validation_data.copy()

validation_predictions["prediction_catboost"] = np.maximum(
    0,
    model.predict(X_valid),
)

methods_with_catboost = BASELINE_METHODS.copy()
methods_with_catboost["CatBoost"] = "prediction_catboost"

model_results = evaluate_forecasts(
    validation_predictions,
    methods_with_catboost,
)

display(
    model_results.style.format({
        "MAE": "{:.3f}",
        "WAPE": "{:.2%}",
        "Bias": "{:+.2%}",
    })
)

baseline_mae = model_results.loc["Среднее за 4 недели", "MAE"]
catboost_mae = model_results.loc["CatBoost", "MAE"]
relative_gain = (baseline_mae - catboost_mae) / baseline_mae

print(f"Снижение MAE относительно среднего 4 недель: {relative_gain:+.2%}")


,MAE,WAPE,Bias
Метод,,,
Предыдущая неделя,4.001,39.33%,-1.16%
Среднее за 4 недели,3.559,34.98%,-2.07%
Медиана за 4 недели,3.618,35.57%,-4.27%
CatBoost,3.372,33.15%,-6.26%


Снижение MAE относительно среднего 4 недель: +5.24%


Проверяем те же метрики отдельно по категориям, чтобы видеть, где модель выигрывает у среднего четырёх недель.


In [25]:
catboost_category_results = pd.concat(
    {
        category: evaluate_forecasts(group, methods_with_catboost)
        for category, group in validation_predictions.groupby(
            "cat_id", observed=True
        )
    },
    names=["category", "Метод"],
)

display(
    catboost_category_results.style.format({
        "MAE": "{:.3f}",
        "WAPE": "{:.2%}",
        "Bias": "{:+.2%}",
    })
)


Осталось 1,188 деревьев. MAE — 3.372, WAPE — 33.15%, Bias — -6.26%. У среднего четырёх недель MAE — 3.559; CatBoost снижает его на 5.24%.

При лимите 1 000 деревьев MAE был 3.378. Более долгое обучение дало небольшую прибавку; повторный запуск с прежним лимитом из ноутбука убрали. Недопрогноз сохраняется. Дальше проверим новые признаки и устойчивость результата на других временных отрезках.


## 6. Новые признаки

Проверяем, что можно добавить к исходным лагам и средним. Даты обучения и валидации сохраняем. Сначала добавляем категории товара, затем более длинную историю продаж.

### 6.1. Категории товара

Передаём `cat_id` и `dept_id` как категориальные признаки. Так модель получает явную принадлежность товара к категории и отделу. Остальные признаки и настройки остаются прежними. После обучения сравниваем результат с исходным CatBoost.


In [26]:
extra_categories = ["cat_id", "dept_id"]

features_categories = [*FEATURE_COLUMNS, *extra_categories]
categorical_categories = [*CATEGORICAL_FEATURES, *extra_categories]

X_train_categories = model_data.loc[
    train_mask, features_categories
].copy()

X_valid_categories = validation_data[features_categories].copy()

for frame in (X_train_categories, X_valid_categories):
    for column in categorical_categories:
        frame[column] = frame[column].astype(str).astype(object)

assert X_train_categories.index.equals(y_train.index)
assert X_valid_categories.index.equals(y_valid.index)

category_params = model.get_params().copy()
category_params["iterations"] = 5000

model_categories = CatBoostRegressor(**category_params)
print(f"Лимит итераций: {model_categories.get_params()['iterations']}")

model_categories.fit(
    X_train_categories,
    y_train,
    cat_features=categorical_categories,
    eval_set=(X_valid_categories, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model_categories.tree_count_}")

Лимит итераций: 5000
0:	learn: 9.8212731	test: 8.4485598	best: 8.4485598 (0)	total: 119ms	remaining: 9m 55s
250:	learn: 4.1059017	test: 3.4258505	best: 3.4258029 (240)	total: 18.5s	remaining: 5m 50s
500:	learn: 4.0494692	test: 3.3940208	best: 3.3940208 (500)	total: 34s	remaining: 5m 5s
750:	learn: 3.9856226	test: 3.3755916	best: 3.3752770 (749)	total: 51.3s	remaining: 4m 49s
1000:	learn: 3.9538363	test: 3.3654827	best: 3.3654323 (982)	total: 1m 9s	remaining: 4m 36s
1250:	learn: 3.9085883	test: 3.3541144	best: 3.3540950 (1230)	total: 1m 28s	remaining: 4m 25s
1500:	learn: 3.8797305	test: 3.3497580	best: 3.3490570 (1470)	total: 1m 48s	remaining: 4m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.34812512
bestIteration = 1612

Shrink model to first 1613 iterations.
Деревьев в модели: 1613


In [27]:
category_predictions = validation_predictions.copy()

category_predictions["prediction_categories"] = np.maximum(
    0,
    model_categories.predict(X_valid_categories),
)

category_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "CatBoost — исходный": "prediction_catboost",
    "CatBoost + категории": "prediction_categories",
}

category_comparison = evaluate_forecasts(
    category_predictions,
    category_methods,
)

display(
    category_comparison.style.format({
        "MAE": "{:.4f}",
        "WAPE": "{:.2%}",
        "Bias": "{:+.2%}",
    })
)

old_mae = category_comparison.loc["CatBoost — исходный", "MAE"]
new_mae = category_comparison.loc["CatBoost + категории", "MAE"]
gain = (old_mae - new_mae) / old_mae

print(f"Снижение MAE после добавления категорий: {gain:+.2%}")

,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
CatBoost — исходный,3.3722,33.15%,-6.26%
CatBoost + категории,3.3471,32.90%,-6.12%


Снижение MAE после добавления категорий: +0.74%


### 6.2. История и направление изменений

Добавляем четыре признака:

- `mean_2w` — средние продажи последних двух недель;
- `mean_13w` — средние продажи последних тринадцати недель;
- `trend_2w_8w` — разница средних за две и восемь недель;
- `std_8w` — стандартное отклонение продаж за восемь недель.

Перед расчётами стоит `shift(1)`: продажи прогнозируемой недели в признаки не попадают. Когда тринадцати прошлых недель ещё нет, оставляем `NaN`. Строки обучения сохраняем; CatBoost обрабатывает числовые пропуски.

Сравниваем новую модель с вариантом, в котором уже есть категории.


In [28]:
history_data = model_data.sort_values(
    ["store_id", "item_id", "week_end"]
).copy()

sales_groups = history_data.groupby(
    ["store_id", "item_id"],
    observed=True,
    sort=False,
)["weekly_sales"]

for window in (2, 13):
    history_data[f"mean_{window}w"] = sales_groups.transform(
        lambda series: series.shift(1).rolling(window).mean()
    )

history_data["std_8w"] = sales_groups.transform(
    lambda series: series.shift(1).rolling(8).std()
)

history_data["trend_2w_8w"] = (
    history_data["mean_2w"] - history_data["mean_8w"]
)

new_history_features = [
    "mean_2w",
    "mean_13w",
    "trend_2w_8w",
    "std_8w",
]

features_history = [*features_categories, *new_history_features]

X_train_history = history_data.loc[
    X_train_categories.index, features_history
].copy()

X_valid_history = history_data.loc[
    X_valid_categories.index, features_history
].copy()

for frame in (X_train_history, X_valid_history):
    for column in categorical_categories:
        frame[column] = frame[column].astype(str).astype(object)

assert X_train_history.index.equals(y_train.index)
assert X_valid_history.index.equals(y_valid.index)

In [29]:
history_params = model_categories.get_params().copy()
history_params["iterations"] = 5000

model_history = CatBoostRegressor(**history_params)
print(f"Лимит итераций: {model_history.get_params()['iterations']}")

model_history.fit(
    X_train_history,
    y_train,
    cat_features=categorical_categories,
    eval_set=(X_valid_history, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model_history.tree_count_}")

Лимит итераций: 5000
0:	learn: 9.8162009	test: 8.4439309	best: 8.4439309 (0)	total: 103ms	remaining: 8m 35s
250:	learn: 4.1243521	test: 3.4153579	best: 3.4153514 (239)	total: 19.1s	remaining: 6m 1s
500:	learn: 3.9838341	test: 3.3638258	best: 3.3638258 (500)	total: 37.5s	remaining: 5m 36s
750:	learn: 3.9486388	test: 3.3530238	best: 3.3523817 (723)	total: 56.5s	remaining: 5m 19s
1000:	learn: 3.9120279	test: 3.3412943	best: 3.3412943 (1000)	total: 1m 16s	remaining: 5m 5s
1250:	learn: 3.8855412	test: 3.3383434	best: 3.3378760 (1187)	total: 1m 36s	remaining: 4m 49s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.33787596
bestIteration = 1187

Shrink model to first 1188 iterations.
Деревьев в модели: 1188


In [30]:
history_predictions = category_predictions.copy()

assert history_predictions.index.equals(X_valid_history.index)

history_predictions["prediction_history"] = np.maximum(
    0,
    model_history.predict(X_valid_history),
)

history_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "CatBoost + категории": "prediction_categories",
    "CatBoost + категории + история": "prediction_history",
}

history_comparison = evaluate_forecasts(
    history_predictions,
    history_methods,
)

display(
    history_comparison.style.format({
        "MAE": "{:.4f}",
        "WAPE": "{:.2%}",
        "Bias": "{:+.2%}",
    })
)

old_mae = history_comparison.loc["CatBoost + категории", "MAE"]
new_mae = history_comparison.loc[
    "CatBoost + категории + история", "MAE"
]

gain = (old_mae - new_mae) / old_mae
print(f"Снижение MAE после добавления истории: {gain:+.2%}")

,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
CatBoost + категории,3.3471,32.90%,-6.12%
CatBoost + категории + история,3.3374,32.81%,-6.49%


Снижение MAE после добавления истории: +0.29%


### 6.3. Отдельные проверки трёх блоков

Дальше используем модель «категории + история» как общую основу. К ней отдельно добавляем нулевые продажи, календарь с SNAP и историю цены. В таблице каждая строка с `+` означает отдельный эксперимент с одним новым блоком.

`run_feature_block` берёт те же строки обучения и валидации, обучает модель и добавляет её прогноз в общую таблицу. Лимит — 5 000 деревьев, ранняя остановка — 100 итераций без улучшения MAE. Положительное снижение MAE означает улучшение относительно модели с историей.


In [31]:
experiment_data = history_data.copy()
experiment_predictions = history_predictions.copy()

experiment_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "Категории + история": "prediction_history",
}


def run_feature_block(name, added_features):
    columns = [*features_history, *added_features]
    categorical = [
        column
        for column in [*categorical_categories, "week_events"]
        if column in columns
    ]

    X_train_block = experiment_data.loc[
        X_train_history.index, columns
    ].copy()

    X_valid_block = experiment_data.loc[
        X_valid_history.index, columns
    ].copy()

    for frame in (X_train_block, X_valid_block):
        for column in categorical:
            frame[column] = frame[column].astype(str).astype(object)

    assert X_train_block.index.equals(y_train.index)
    assert X_valid_block.index.equals(y_valid.index)
    assert experiment_predictions.index.equals(X_valid_block.index)

    params = model_history.get_params().copy()
    params["iterations"] = 5000

    candidate = CatBoostRegressor(**params)
    candidate.fit(
        X_train_block,
        y_train,
        cat_features=categorical,
        eval_set=(X_valid_block, y_valid),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=500,
    )

    experiment_predictions[name] = np.maximum(
        0, candidate.predict(X_valid_block)
    )
    experiment_methods[name] = name

    results = evaluate_forecasts(
        experiment_predictions, experiment_methods
    )

    display(results.style.format({
        "MAE": "{:.4f}",
        "WAPE": "{:.2%}",
        "Bias": "{:+.2%}",
    }))

    base_mae = results.loc["Категории + история", "MAE"]
    gain = 1 - results.loc[name, "MAE"] / base_mae

    print(f"Снижение MAE относительно модели с историей: {gain:+.2%}")
    print(f"Деревьев сохранено: {candidate.tree_count_}")

    return candidate

### 6.4. Нулевые продажи

Описываем, насколько регулярно покупают товар:

- `zero_share_8w` — доля нулевых недель среди последних восьми;
- `zero_run` — число последних недель подряд без продаж;
- `mean_nonzero_8w` — средние продажи в ненулевые недели за то же окно.

Все значения считаются по прошлым продажам. Если за восемь недель положительных продаж не было, среднее по ненулевым неделям остаётся `NaN`. Причину отсутствия продаж эти признаки не определяют.


In [32]:
def zero_share(series):
    past = series.shift(1)
    zeros = past.eq(0).astype(float).where(past.notna())
    return zeros.rolling(8).mean()


def zero_run(series):
    zeros = series.shift(1).eq(0)
    runs = (~zeros).cumsum()
    return zeros.groupby(runs).cumsum()


sales_groups = experiment_data.groupby(
    ["store_id", "item_id"],
    observed=True,
    sort=False,
)["weekly_sales"]

experiment_data["zero_share_8w"] = sales_groups.transform(zero_share)
experiment_data["zero_run"] = sales_groups.transform(zero_run)

experiment_data["mean_nonzero_8w"] = sales_groups.transform(
    lambda series: (
        series.shift(1)
        .replace(0, np.nan)
        .rolling(8, min_periods=1)
        .mean()
    )
)

zero_features = [
    "zero_share_8w",
    "zero_run",
    "mean_nonzero_8w",
]

model_zeros = run_feature_block("+ нулевые продажи", zero_features)

0:	learn: 9.8379520	test: 8.4629858	best: 8.4629858 (0)	total: 105ms	remaining: 8m 46s
500:	learn: 4.0428246	test: 3.3777779	best: 3.3777767 (499)	total: 42.4s	remaining: 6m 21s
1000:	learn: 3.9542583	test: 3.3534882	best: 3.3534882 (1000)	total: 1m 19s	remaining: 5m 15s
1500:	learn: 3.8880475	test: 3.3366246	best: 3.3366246 (1498)	total: 2m 1s	remaining: 4m 42s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.333938337
bestIteration = 1613

Shrink model to first 1614 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%


Снижение MAE относительно модели с историей: +0.13%
Деревьев сохранено: 1614


### 6.5. События календаря и SNAP

Для прогнозируемой недели собираем:

- `snap_days` — число дней с флагом `snap_CA`;
- `event_days` — число дней хотя бы с одним событием;
- `week_events` — названия событий за неделю, категориальный признак.

Эта информация берётся из календаря и известна заранее. Дневные признаки собираем в недельные, поскольку прогнозируем сумму продаж с понедельника по воскресенье.


In [33]:
calendar_days = calendar.copy()

calendar_days["week_end"] = (
    calendar_days["date"]
    + pd.to_timedelta(
        6 - calendar_days["date"].dt.dayofweek,
        unit="D",
    )
)

calendar_days["has_event"] = (
    calendar_days["event_name_1"].notna()
    | calendar_days["event_name_2"].notna()
).astype("int8")

calendar_weekly = calendar_days.groupby("week_end").agg(
    snap_days=("snap_CA", "sum"),
    event_days=("has_event", "sum"),
)

events = calendar_days.melt(
    id_vars="week_end",
    value_vars=["event_name_1", "event_name_2"],
    value_name="event",
).dropna(subset=["event"])

calendar_weekly["week_events"] = events.groupby("week_end")["event"].agg(
    lambda names: " | ".join(sorted(names.unique()))
)
calendar_weekly["week_events"] = calendar_weekly["week_events"].fillna(
    "NoEvent"
)

calendar_features = ["snap_days", "event_days", "week_events"]

experiment_data = experiment_data.join(
    calendar_weekly[calendar_features],
    on="week_end",
    validate="many_to_one",
)

model_calendar = run_feature_block(
    "+ календарь и SNAP", calendar_features
)

0:	learn: 9.8267904	test: 8.4535844	best: 8.4535844 (0)	total: 117ms	remaining: 9m 46s
500:	learn: 3.9790332	test: 3.3795263	best: 3.3794671 (457)	total: 51.1s	remaining: 7m 38s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.379467058
bestIteration = 457

Shrink model to first 458 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%


Снижение MAE относительно модели с историей: -1.24%
Деревьев сохранено: 458


### 6.6. История цены

Ценовая неделя M5 идёт с субботы по пятницу. Для прогноза, который делаем в воскресенье, используем цены уже завершившихся периодов. Сдвиг на девять дней связывает пятницу ценового периода с воскресеньем прогнозируемой недели.

Добавляем:

- `price_last` — последнюю известную цену завершённого периода;
- `price_change_1w` — относительное изменение этой цены за неделю;
- `price_relative_4w` — отклонение от средней цены за четыре периода;
- `price_missing_last_period` — отсутствие записи о цене за последний завершённый период.

Пропуски цены заполняем только предыдущими известными значениями через `ffill`. Отсутствие записи о цене не означает отсутствие товара на складе.


In [34]:
series_keys = ["store_id", "item_id"]

price_week_ends = (
    calendar.groupby("wm_yr_wk")["date"].min()
    + pd.Timedelta(days=6)
)

price_lookup = prices[
    [*series_keys, "wm_yr_wk", "sell_price"]
].copy()

price_lookup["week_end"] = (
    price_lookup["wm_yr_wk"].map(price_week_ends)
    + pd.Timedelta(days=9)
)

price_lookup = price_lookup.rename(
    columns={"sell_price": "price_quote"}
).set_index([*series_keys, "week_end"])

experiment_data = experiment_data.join(
    price_lookup[["price_quote"]],
    on=[*series_keys, "week_end"],
    validate="many_to_one",
)

experiment_data["price_missing_last_period"] = (
    experiment_data["price_quote"].isna().astype("int8")
)

experiment_data["price_last"] = experiment_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_quote"].ffill()

price_groups = experiment_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_last"]

previous_price = price_groups.shift(1)
mean_price_4w = price_groups.transform(
    lambda series: series.rolling(4, min_periods=1).mean()
)

experiment_data["price_change_1w"] = (
    experiment_data["price_last"] / previous_price - 1
)
experiment_data["price_relative_4w"] = (
    experiment_data["price_last"] / mean_price_4w - 1
)

price_features = [
    "price_last",
    "price_change_1w",
    "price_relative_4w",
    "price_missing_last_period",
]

model_price = run_feature_block("+ история цены", price_features)

0:	learn: 9.8125777	test: 8.4399539	best: 8.4399539 (0)	total: 110ms	remaining: 9m 8s
500:	learn: 3.9716664	test: 3.3668528	best: 3.3668528 (500)	total: 39.7s	remaining: 5m 56s
1000:	learn: 3.8969480	test: 3.3413291	best: 3.3413291 (1000)	total: 1m 21s	remaining: 5m 24s
1500:	learn: 3.8540552	test: 3.3335833	best: 3.3327510 (1420)	total: 2m 2s	remaining: 4m 46s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.332751016
bestIteration = 1420

Shrink model to first 1421 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%
+ история цены,3.3322,32.76%,-6.44%


Снижение MAE относительно модели с историей: +0.15%
Деревьев сохранено: 1421


### 6.7. Результат отдельных экспериментов

На выбранных восьми неделях получили:

| Вариант | MAE | Снижение MAE относительно модели с историей |
|---|---:|---:|
| Категории + история | 3.3374 | — |
| + нулевые продажи | 3.3331 | +0.13% |
| + календарь и SNAP | 3.3787 | −1.24% |
| + история цены | 3.3322 | +0.15% |

Нулевые продажи и цена дали небольшое улучшение. Календарь с SNAP ухудшил результат относительно модели с историей, хотя всё ещё обогнал среднее четырёх недель.

Следующий эксперимент — обучить модель со всеми блоками вместе. Отдельные улучшения не гарантируют улучшение общего набора. После совместной проверки перейдём к подбору гиперпараметров и оценке на нескольких временных окнах.


In [35]:
all_added_features = list(dict.fromkeys([
    *zero_features,
    *calendar_features,
    *price_features,
]))

features_all = [*features_history, *all_added_features]

print(f"Всего признаков: {len(features_all)}")

model_all_features = run_feature_block(
    "+ все признаки",
    all_added_features,
)

Всего признаков: 27
0:	learn: 9.8217103	test: 8.4476763	best: 8.4476763 (0)	total: 144ms	remaining: 11m 57s
500:	learn: 3.9490344	test: 3.3453014	best: 3.3453014 (500)	total: 51.3s	remaining: 7m 40s
1000:	learn: 3.8511589	test: 3.3209769	best: 3.3209769 (1000)	total: 1m 45s	remaining: 7m 1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.320323451
bestIteration = 1126

Shrink model to first 1127 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%
+ история цены,3.3322,32.76%,-6.44%
+ все признаки,3.3192,32.63%,-5.39%


Снижение MAE относительно модели с историей: +0.54%
Деревьев сохранено: 1127
